# SKF Observer Phoenix — Notebook de Testes
> Sandbox para validar funções, parsers e plots do dashboard Streamlit.  
> Cada seção replica uma aba do app e pode ser executada de forma independente.

**Índice**
1. [Configuração e Autenticação](#1-configuração-e-autenticação)
2. [Assets e Points](#2-assets-e-points)
3. [Tendência — Parse & Plot](#3-tendência--parse--plot)
4. [Espectro FFT — Parse & Plot](#4-espectro-fft--parse--plot)
5. [IMx-1 Comissionamento](#5-imx-1-comissionamento)
6. [Fleet Monitoring — Gateways & Sensores](#6-fleet-monitoring--gateways--sensores)
7. [Fleet — Dispositivos Cabeados & System Check](#7-fleet--dispositivos-cabeados--system-check)


## 1. Configuração e Autenticação

In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timezone, timedelta
import time
import json

# ── Paleta ───────────────────────────────
C_BG       = "#0e1820"
C_PANEL    = "#162130"
C_BORDER   = "#2a3f52"
C_ACCENT   = "#A7C5E2"   # Blue claro
C_SOLID    = "#32556E"   # Blue sólido
C_DEEP     = "#007CAA"   # Azul vivo
C_OK       = "#4E9D2D"   # Green
C_WARN     = "#BA944B"   # Âmbar
C_DANGER   = "#F06A22"   # Laranja crítico
C_TEAL     = "#379A8D"   # Teal
C_PURPLE   = "#5E699E"   # Roxo secundário
C_TEXT     = "#EEF4F9"
C_TEXT_MID = "#98C0B8"
C_GRID     = "#2a3f52"

PLOTLY_LAYOUT = dict(
    paper_bgcolor=C_BG,
    plot_bgcolor=C_PANEL,
    font=dict(family="sans-serif", color=C_TEXT_MID, size=11),
    xaxis=dict(gridcolor=C_GRID, zeroline=False,
               tickfont=dict(size=10, color=C_TEXT_MID)),
    yaxis=dict(gridcolor=C_GRID, zeroline=False,
               tickfont=dict(size=10, color=C_TEXT_MID)),
    legend=dict(bgcolor="rgba(22,33,48,0.9)", bordercolor=C_BORDER,
                borderwidth=1, font=dict(size=10, color=C_ACCENT)),
    hovermode="x unified",
    margin=dict(l=60, r=40, t=60, b=50),
)

print("✅ Imports e paleta OK")


In [ ]:
# ── Unidades disponíveis ─────────────────────
UNITS = {
    "ABC":      "http://services.repcenter.skf.com:",
    "ABCD":          "http://services.repcenter.skf.com:",
    "ABCDE":              "http://services.repcenter.skf.com:",
    "ABCDEF": "http://services.repcenter.skf.com:",
    "ABCDEFG":       "http://services.repcenter.skf.com:",
}

# ── Selecione a unidade e credenciais ────────
UNIT_NAME = "ABC"          # ← altere aqui
BASE_URL  = UNITS[UNIT_NAME]
USERNAME  = "patrick.coelho"
PASSWORD  = ""                        # ← preencha antes de executar

TIMEOUT = 30

print(f"Unidade : {UNIT_NAME}")
print(f"URL     : {BASE_URL}")


In [ ]:
# ── Autenticação ─────────────────────────────
def autenticar(base_url, username, password):
    resp = requests.post(
        f"{base_url}/token",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "password", "username": username, "password": password},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

TOKEN      = autenticar(BASE_URL, USERNAME, PASSWORD)
TOKEN_TIME = time.time()
print(f"✅ Token obtido: {TOKEN[:30]}…")


In [ ]:
# ── Renovação automática de token ────────────
TOKEN_VALIDITY = 1140   # 19 minutos

def ensure_token():
    global TOKEN, TOKEN_TIME
    if time.time() - TOKEN_TIME > TOKEN_VALIDITY:
        print("🔄 Renovando token…")
        TOKEN      = autenticar(BASE_URL, USERNAME, PASSWORD)
        TOKEN_TIME = time.time()
    return TOKEN

def headers():
    return {"Authorization": f"Bearer {ensure_token()}",
            "Accept": "application/json"}

print("✅ ensure_token() pronto")


---
## 2. Assets e Points

In [ ]:
# ── GET /v2/assets ───────────────────────────
def get_assets():
    resp = requests.get(f"{BASE_URL}/v2/assets",
                        headers=headers(), timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

assets = get_assets()
df_assets = pd.DataFrame([{
    "ID":     a.get("ID"),
    "Nome":   a.get("Name"),
    "Path":   a.get("Path"),
    "Status": a.get("Status"),
} for a in assets])

print(f"✅ {len(df_assets)} asset(s) encontrado(s)")
df_assets.head(10)


In [ ]:
# ── Selecione um asset para explorar ─────────
# Veja df_assets acima e preencha o ID desejado
ASSET_ID   = df_assets.iloc[0]["ID"]   # ← ou coloque o ID diretamente
ASSET_NAME = df_assets.iloc[0]["Nome"]
print(f"Asset selecionado: [{ASSET_ID}] {ASSET_NAME}")


In [ ]:
# ── GET /v2/points ────────────────────────────
def get_points(asset_id):
    resp = requests.get(f"{BASE_URL}/v2/points",
                        headers=headers(),
                        params={"machine_id": asset_id},
                        timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

points = get_points(ASSET_ID)
df_points = pd.DataFrame([{
    "ID":   p.get("id") or p.get("ID"),
    "Nome": p.get("name") or p.get("Name"),
    "Tipo": p.get("type") or p.get("Type"),
    "EU":   p.get("unit") or p.get("Unit") or p.get("eu"),
} for p in points])

print(f"✅ {len(df_points)} point(s) encontrado(s)")
df_points


In [ ]:
# ── Selecione um point para explorar ─────────
POINT_ID = df_points.iloc[0]["ID"]   # ← ou coloque o ID diretamente
print(f"Point selecionado: [{POINT_ID}] {df_points.iloc[0]['Nome']}")


---
## 3. Tendência — Parse & Plot

In [ ]:
# ── GET /v1/points/{id}/trendMeasurements ────
FROM_DATE = "2026-01-01T00:00:00Z"
TO_DATE   = "2026-03-31T23:59:59Z"
MAX_READ  = 500

def get_trend(point_id, from_date=FROM_DATE, to_date=TO_DATE, max_readings=MAX_READ):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/trendMeasurements",
        headers=headers(),
        params={
            "pointId":            point_id,
            "maxNumberOfReadings": max_readings,
            "fromDateUTC":        from_date,
            "toDateUTC":          to_date,
            "descending":         "false",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json() if resp.status_code != 204 else []

raw_trend = get_trend(POINT_ID)
print(f"✅ {len(raw_trend)} leitura(s) retornada(s)")
# Inspeciona a primeira leitura
if raw_trend:
    print(json.dumps(raw_trend[0], indent=2, default=str))


In [ ]:
# ── parse_trend ──────────────────────────────
def parse_trend(data):
    rows = []
    for item in data:
        if "Measurements" in item and isinstance(item["Measurements"], list):
            ts_raw  = (item.get("ReadingTimeUTC") or item.get("ReadingTime")
                       or item.get("dateUTC") or item.get("timestamp"))
            speed   = item.get("Speed")
            process = item.get("Process")
            for meas in item["Measurements"]:
                rows.append({
                    "timestamp":    ts_raw,
                    "value":        meas.get("Level"),
                    "unit":         meas.get("Units", ""),
                    "channel":      meas.get("Channel", 1),
                    "channel_name": meas.get("ChannelName", "Overall"),
                    "direction":    meas.get("Direction", ""),
                    "bov":          meas.get("BOV"),
                    "speed":        speed,
                    "process":      process,
                })
        else:
            ts_raw  = (item.get("timestamp") or item.get("dateUTC")
                       or item.get("ReadingTimeUTC"))
            val_obj = item.get("value") or {}
            value   = val_obj.get("value") if isinstance(val_obj, dict) else val_obj
            unit    = val_obj.get("unit", "") if isinstance(val_obj, dict) else item.get("unit", "")
            rows.append({
                "timestamp": ts_raw, "value": value, "unit": unit,
                "channel": 1, "channel_name": "Overall",
                "direction": "", "bov": None, "speed": None, "process": None,
            })
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df["value"]     = pd.to_numeric(df["value"],     errors="coerce")
    df["speed"]     = pd.to_numeric(df["speed"],     errors="coerce")
    df.dropna(subset=["timestamp", "value"], inplace=True)
    df.sort_values("timestamp", inplace=True)
    df.drop_duplicates(subset=["timestamp", "channel"], keep="last", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

df_trend = parse_trend(raw_trend)
print(f"✅ DataFrame: {len(df_trend)} linhas, {df_trend['channel'].nunique()} canal(is)")
print(f"   Período : {df_trend['timestamp'].min()} → {df_trend['timestamp'].max()}")
print(f"   Unidade : {df_trend['unit'].iloc[0] if len(df_trend) else '—'}")
df_trend.head()


In [ ]:
# ── Seletor de canal ──────────────────────────
channels = df_trend.groupby("channel")["channel_name"].first().to_dict()
print("Canais disponíveis:")
for ch, name in channels.items():
    print(f"  Canal {ch}: {name}")

# Selecione o canal para plotar
CHANNEL = list(channels.keys())[0]
df_ch   = df_trend[df_trend["channel"] == CHANNEL].copy()
unit    = df_ch["unit"].iloc[0]
print(f"\nPlotando canal {CHANNEL} — {channels[CHANNEL]} [{unit}]")


In [ ]:
# ── Plot de Tendência ─────────────────────────
mu    = df_ch["value"].mean()
sigma = df_ch["value"].std()

fig = go.Figure()

# Área sob a linha
fig.add_trace(go.Scatter(
    x=df_ch["timestamp"], y=df_ch["value"],
    fill="tozeroy", fillcolor=f"rgba(50,85,110,0.08)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
))

# Linha principal
fig.add_trace(go.Scatter(
    x=df_ch["timestamp"], y=df_ch["value"],
    mode="lines", name=f"{channels[CHANNEL]} [{unit}]",
    line=dict(color=C_ACCENT, width=1.8),
    hovertemplate="%{x|%d/%m %H:%M}<br><b>%{y:.3f}</b> " + unit + "<extra></extra>",
))

# Banda ±1σ
fig.add_trace(go.Scatter(
    x=pd.concat([df_ch["timestamp"], df_ch["timestamp"][::-1]]),
    y=pd.concat([pd.Series([mu + sigma] * len(df_ch)),
                 pd.Series([mu - sigma] * len(df_ch[::-1]))]),
    fill="toself", fillcolor="rgba(50,85,110,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="±1σ", showlegend=True,
))

# Linhas de referência
fig.add_hline(y=mu,             line=dict(color=C_OK,     width=1,   dash="dash"),
              annotation_text=f"μ = {mu:.3f}", annotation_font=dict(color=C_OK, size=9))
fig.add_hline(y=mu + 2 * sigma, line=dict(color=C_WARN,   width=1,   dash="dot"),
              annotation_text="Alerta (μ+2σ)",  annotation_font=dict(color=C_WARN, size=9))
fig.add_hline(y=mu + 3 * sigma, line=dict(color=C_DANGER, width=1.2, dash="dot"),
              annotation_text="Alarme (μ+3σ)",  annotation_font=dict(color=C_DANGER, size=9))

# Velocidade no eixo secundário (se disponível)
df_spd = df_ch.dropna(subset=["speed"])
if not df_spd.empty and df_spd["speed"].max() > 0:
    fig.add_trace(go.Scatter(
        x=df_spd["timestamp"], y=df_spd["speed"],
        mode="lines", name="Velocidade (RPM)",
        yaxis="y2", line=dict(color=C_PURPLE, width=1.2, dash="dot"),
    ))
    fig.update_layout(yaxis2=dict(
        title=dict(text="RPM", font=dict(color=C_PURPLE, size=10)),
        overlaying="y", side="right", gridcolor="rgba(0,0,0,0)",
        tickfont=dict(size=9, color=C_PURPLE),
    ))

layout = PLOTLY_LAYOUT.copy()
layout.update(dict(
    title=dict(text=f"<b>Tendência</b> · Canal {CHANNEL} — {channels[CHANNEL]} [{unit}]",
               font=dict(size=16, color=C_TEXT), x=0.01),
    xaxis_title="Data/Hora",
    yaxis_title=f"Nível [{unit}]",
    height=420,
))
fig.update_layout(**layout)
fig.show()
print(f"  μ={mu:.4f}  σ={sigma:.4f}  min={df_ch['value'].min():.4f}  max={df_ch['value'].max():.4f}")


---
## 4. Espectro FFT — Parse & Plot

In [ ]:
# ── GET /v1/points/{id}/dynamicMeasurements ──
def get_spectrum(point_id):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/dynamicMeasurements",
        headers=headers(), timeout=30,
    )
    if resp.status_code in (204, 404):
        return None
    resp.raise_for_status()
    return resp.json()

raw_spec = get_spectrum(POINT_ID)
if raw_spec:
    # Mostra metadados sem os Values (array muito longo)
    preview = {k: v for k, v in (raw_spec if isinstance(raw_spec, dict)
               else raw_spec[0]).items() if k != "Measurements"}
    print(json.dumps(preview, indent=2, default=str))
    meas = (raw_spec if isinstance(raw_spec, dict) else raw_spec[0]).get("Measurements", [])
    for i, m in enumerate(meas):
        print(f"  Canal {i}: Direction={m.get('Direction')}  "
              f"MeasurementType={m.get('MeasurementType')}  "
              f"len(Values)={len(m.get('Values', []))}")
else:
    print("⚠ Ponto sem medição dinâmica (espectro)")


In [ ]:
# ── parse_spectrum ────────────────────────────
DIRECTION_MAP = {0: "X", 1: "Y", 2: "Z", 3: "H", 4: "V", 5: "A"}
MTYPE_MAP     = {0: "Waveform", 1: "Espectro (velocidade)", 2: "Espectro", 3: "Cepstrum"}

def parse_spectrum(data):
    if data is None:
        return []
    items = data if isinstance(data, list) else [data]
    channels = []
    for item in items:
        eu          = item.get("EU") or item.get("EUSpectrum") or "—"
        start_f     = float(item.get("StartFrequency", 0.0))
        end_f       = float(item.get("EndFrequency",   0.0))
        sample_rate = float(item.get("SampleRate",     0.0))
        samples     = int(item.get("Samples",          0))
        speed       = float(item.get("Speed", 0) or 0)
        speed_units = item.get("SpeedUnits", "RPM")
        for idx, meas in enumerate(item.get("Measurements") or []):
            values = meas.get("Values") or []
            if not values:
                continue
            n     = len(values)
            freqs = np.linspace(start_f, end_f, n) if end_f > start_f else np.arange(n)
            channels.append({
                "channel_idx": idx,
                "direction":   meas.get("Direction", idx),
                "mtype":       meas.get("MeasurementType", 0),
                "eu":          eu,
                "start_freq":  start_f,
                "end_freq":    end_f,
                "freqs":       np.array(freqs, dtype=float),
                "values":      np.array(values, dtype=float),
                "sample_rate": sample_rate,
                "samples":     samples,
                "speed":       speed,
                "speed_units": speed_units,
            })
    return channels

channels_spec = parse_spectrum(raw_spec)
print(f"✅ {len(channels_spec)} canal(is) parseado(s)")
for ch in channels_spec:
    d = DIRECTION_MAP.get(ch['direction'], str(ch['direction']))
    m = MTYPE_MAP.get(ch['mtype'], f"Tipo {ch['mtype']}")
    print(f"  Canal {ch['channel_idx']}: {d} | {m} | {ch['eu']} | {len(ch['values'])} linhas "
          f"| {ch['start_freq']:.0f}–{ch['end_freq']:.0f} Hz")


In [ ]:
# ── Plot de Espectro ──────────────────────────
if not channels_spec:
    print("⚠ Sem dados de espectro para plotar")
else:
    CH_IDX = 0   # ← altere para selecionar outro canal
    ch     = channels_spec[CH_IDX]
    freqs  = ch["freqs"]
    values = ch["values"]
    eu     = ch["eu"]
    dir_s  = DIRECTION_MAP.get(ch["direction"], str(ch["direction"]))
    speed  = ch["speed"]

    # Top 10 picos
    peak_idx  = np.argsort(values)[::-1][:10]
    peak_f    = freqs[peak_idx]
    peak_v    = values[peak_idx]

    # Harmônicas de rotação
    harm_lines = []
    if speed and speed > 0:
        rpm_hz = speed / 60.0
        for h in range(1, 6):
            hf = rpm_hz * h
            if ch["start_freq"] <= hf <= ch["end_freq"]:
                harm_lines.append((hf, f"{h}X {hf:.1f}Hz"))

    fig = go.Figure()

    # Área
    fig.add_trace(go.Scatter(
        x=freqs, y=values, fill="tozeroy",
        fillcolor=f"rgba(50,85,110,0.08)",
        line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
    ))
    # Linha
    fig.add_trace(go.Scatter(
        x=freqs, y=values, mode="lines",
        name=f"Espectro [{dir_s}]",
        line=dict(color=C_ACCENT, width=1.5),
        hovertemplate="<b>%{x:.2f} Hz</b><br>%{y:.5f} " + eu + "<extra></extra>",
    ))
    # Picos
    fig.add_trace(go.Scatter(
        x=peak_f, y=peak_v, mode="markers+text",
        name="Picos", marker=dict(size=8, color=C_WARN, symbol="triangle-up"),
        text=[f"{f:.1f}" for f in peak_f],
        textposition="top center",
        textfont=dict(size=8, color=C_WARN),
        hovertemplate="<b>%{x:.2f} Hz</b><br>%{y:.5f} " + eu + "<extra>Pico</extra>",
    ))
    # Harmônicas
    for hf, hlabel in harm_lines:
        fig.add_vline(x=hf, line=dict(color=f"rgba(94,105,158,0.6)", width=1, dash="dot"),
                      annotation_text=hlabel, annotation_font=dict(size=8, color=C_PURPLE))

    layout = PLOTLY_LAYOUT.copy()
    layout.update(dict(
        title=dict(text=f"<b>Espectro FFT</b> · Direção {dir_s} [{eu}]",
                   font=dict(size=16, color=C_TEXT), x=0.01),
        xaxis=dict(**PLOTLY_LAYOUT["xaxis"],
                   title=dict(text="Frequência [Hz]", font=dict(color=C_TEXT_MID)),
                   rangeslider=dict(visible=True, bgcolor=C_PANEL, thickness=0.05)),
        yaxis=dict(**PLOTLY_LAYOUT["yaxis"],
                   title=dict(text=f"Amplitude [{eu}]", font=dict(color=C_TEXT_MID))),
        height=450,
    ))
    fig.update_layout(**layout)
    fig.show()

    print(f"  Pico máximo: {peak_f[0]:.2f} Hz → {peak_v[0]:.5f} {eu}")


---
## 5. IMx-1 Comissionamento

In [ ]:
# ── Constantes IMx-1 ─────────────────────────
IMX1_NODE_TYPES    = {11101, 11102, 11103, 11104}
IMX1_TEMP_NODE_TYPE = 11104
IMX1_TEMP_EU_TYPE   = 10905
COMMISSIONING_FROM  = "2024-05-01T00:00:00"

# ── GET /v1/machines/{id}/points ─────────────
def get_points_v1(machine_id):
    resp = requests.get(f"{BASE_URL}/v1/machines/{machine_id}/points",
                        headers=headers(), timeout=TIMEOUT)
    if resp.status_code in (204, 404):
        return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

# ── GET /v1/nextgensensor (apenas comissionados) ──
def get_imx_sensors():
    resp = requests.get(f"{BASE_URL}/v1/nextgensensor",
                        headers=headers(), timeout=15)
    if resp.status_code in (204, 404):
        return {}
    resp.raise_for_status()
    data  = resp.json()
    items = data if isinstance(data, list) else data.get("value", data.get("items", []))
    index = {}
    for s in items:
        comm = s.get("Commissioned") if s.get("Commissioned") is not None else s.get("commissioned")
        if not comm:
            continue
        nid = s.get("IDNode") or s.get("idNode")
        if nid is not None:
            index[int(nid)] = s
    return index

# ── GET 1ª leitura de tendência ───────────────
def get_first_reading(point_id):
    resp = requests.get(
        f"{BASE_URL}/v1/points/{point_id}/trendMeasurements",
        headers=headers(),
        params={"fromDateUTC": COMMISSIONING_FROM, "numReadings": 1, "descending": "false"},
        timeout=30,
    )
    if resp.status_code in (204, 404):
        return None
    resp.raise_for_status()
    data = resp.json()
    lst  = data if isinstance(data, list) else data.get("value", [])
    return lst[0] if lst else None

print("✅ Funções IMx-1 definidas")


In [ ]:
# ── Varredura de um único asset (teste rápido) ──
# Use ASSET_ID já definido na seção 2
print(f"Varrendo asset [{ASSET_ID}] {ASSET_NAME}…")

sensor_index = get_imx_sensors()
print(f"  {len(sensor_index)} sensor(es) comissionado(s) indexados")

pts_v1    = get_points_v1(ASSET_ID)
imx_pts   = [p for p in pts_v1 if int(p.get("NodeType", 0)) in IMX1_NODE_TYPES]
print(f"  {len(imx_pts)} point(s) IMx-1 encontrado(s)")

# Agrupa por IDNode
nodes = {}
for p in imx_pts:
    nid = p.get("ParentID") or p.get("IDNode") or p.get("NodeID")
    try:   nid = int(nid)
    except: nid = None
    if nid: nodes.setdefault(nid, []).append(p)

print(f"  {len(nodes)} IDNode(s) distintos: {list(nodes.keys())}")


In [ ]:
# ── Coleta completa para os nodes encontrados ──
today = datetime.now(timezone.utc)
rows  = []

for id_node, node_pts in nodes.items():
    if id_node not in sensor_index:
        print(f"  ⊘ IDNode {id_node} — não comissionado")
        continue

    # Melhor point (temperatura > outro)
    temp_pts = [p for p in node_pts
                if int(p.get("NodeType", 0)) == IMX1_TEMP_NODE_TYPE
                or int(p.get("EUType",   0)) == IMX1_TEMP_EU_TYPE]
    target = temp_pts[0] if temp_pts else node_pts[0]
    pid    = target.get("ID") or target.get("id")

    first  = get_first_reading(pid)
    time.sleep(0.15)

    ts_raw = (first.get("ReadingTimeUTC") or first.get("readingTimeUTC")
              if first else None)
    try:   commissioning_dt = pd.to_datetime(ts_raw, utc=True)
    except: commissioning_dt = None

    meta       = sensor_index.get(id_node, {})
    hw_id      = meta.get("SensorIdentifier") or "—"
    battery    = meta.get("BatteryLevel")
    try:   battery = float(battery)
    except: battery = None

    # Regra ClearedDate (sentinela ano ≤ 1940)
    cleared_raw = (meta.get("ClearedDate") or meta.get("clearedDate")
                   or meta.get("Cleared"))
    try:   cleared_dt = pd.to_datetime(cleared_raw, utc=True) if cleared_raw else None
    except: cleared_dt = None
    if cleared_dt and cleared_dt.year <= 1940:
        cleared_dt = None

    if cleared_dt:
        effective_dt = cleared_dt
        fonte        = "ClearedDate"
    elif commissioning_dt:
        effective_dt = commissioning_dt
        fonte        = "1ª Leitura"
    else:
        effective_dt = None
        fonte        = "—"

    dias_uso     = (today - effective_dt).days if effective_dt else None
    taxa_bateria = round((100 - battery) / dias_uso, 4)                    if dias_uso and dias_uso > 0 and battery is not None else None

    rows.append({
        "HardwareID":    hw_id,
        "IDNode":        id_node,
        "MachineID":     ASSET_ID,
        "MachineName":   ASSET_NAME,
        "BatteryLevel":  battery,
        "DataPrimeiraLeitura": commissioning_dt.strftime("%Y-%m-%d %H:%M") if commissioning_dt else None,
        "DataClearedSensor":   cleared_dt.strftime("%Y-%m-%d %H:%M") if cleared_dt else None,
        "ProvavelDataComissionamento": effective_dt.strftime("%Y-%m-%d %H:%M") if effective_dt else None,
        "FonteComissionamento": fonte,
        "DiasDeUso":       dias_uso,
        "TaxaConsumoBateria": taxa_bateria,
    })
    print(f"  ✓ IDNode {id_node} | HW: {hw_id} | Bat: {battery}% | "
          f"Comissionado: {effective_dt} ({fonte})")

df_imx = pd.DataFrame(rows)
print(f"\n✅ DataFrame IMx: {len(df_imx)} sensor(es)")
df_imx


In [ ]:
# ── Plot: Bateria × Dias em Campo ─────────────
if len(df_imx) >= 2:
    df_p = df_imx.dropna(subset=["BatteryLevel", "DiasDeUso"])

    def bat_color(b):
        return C_DANGER if b < 20 else (C_WARN if b < 40 else C_OK)

    fig = go.Figure()
    fig.add_hrect(y0=0,  y1=20,  fillcolor=f"rgba(240,106,34,0.07)", line_width=0)
    fig.add_hrect(y0=20, y1=40,  fillcolor=f"rgba(186,148,75,0.05)", line_width=0)
    fig.add_hrect(y0=40, y1=100, fillcolor=f"rgba(78,157,45,0.03)",  line_width=0)
    fig.add_trace(go.Scatter(
        x=df_p["DiasDeUso"], y=df_p["BatteryLevel"],
        mode="markers",
        marker=dict(size=11, color=[bat_color(b) for b in df_p["BatteryLevel"]],
                    line=dict(color="rgba(255,255,255,0.15)", width=1)),
        text=df_p["MachineName"],
        customdata=df_p[["HardwareID", "TaxaConsumoBateria"]].values,
        hovertemplate="<b>%{text}</b><br>HW: %{customdata[0]}<br>"
                      "Bateria: %{y:.1f}%<br>Dias: %{x}<br>"
                      "Taxa: %{customdata[1]:.4f}%/dia<extra></extra>",
    ))
    layout = PLOTLY_LAYOUT.copy()
    layout.update(dict(
        title=dict(text="<b>Bateria × Dias em Campo</b>",
                   font=dict(size=16, color=C_TEXT), x=0.01),
        xaxis=dict(**PLOTLY_LAYOUT["xaxis"],
                   title=dict(text="Dias em Campo", font=dict(color=C_TEXT_MID))),
        yaxis=dict(**PLOTLY_LAYOUT["yaxis"], range=[0, 105],
                   title=dict(text="Bateria [%]", font=dict(color=C_TEXT_MID))),
        height=380,
    ))
    fig.update_layout(**layout)
    fig.show()
else:
    print("ℹ Poucos dados para o scatter — rode o scan completo acima com mais assets.")


---
## 6. Fleet Monitoring — Gateways & Sensores

In [ ]:
# ── Constantes OPC UA ────────────────────────
CONN_STATE_LABELS = {
    0: "Desconectado",
    1: "Conectado",
    2: "Sem Medição",
    3: "Conectado — Sem Medição",
}
DIAG_BITS = {1: "Bateria Baixa", 512: "Instabilidade de Rede"}

def _parse_dt(raw):
    if not raw: return None
    try:   return pd.to_datetime(raw, utc=True)
    except: return None

# ── GET /v1/gateways ──────────────────────────
def get_gateways():
    resp = requests.get(f"{BASE_URL}/v1/gateways",
                        headers=headers(), timeout=15)
    if resp.status_code in (204, 404): return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

today_utc = datetime.now(timezone.utc)
gw_raw    = get_gateways()
gw_rows   = []
for g in gw_raw:
    updated  = _parse_dt(g.get("statusLastUpdated") or g.get("StatusLastUpdated"))
    connected = g.get("connected") if g.get("connected") is not None else g.get("Connected")
    gw_rows.append({
        "GatewayID":         g.get("id") or g.get("ID"),
        "Nome":              g.get("name") or g.get("Name") or "—",
        "Conectado":         bool(connected),
        "StatusLastUpdated": updated.strftime("%Y-%m-%d %H:%M:%S") if updated else None,
        "DiasSemUpdate":     (today_utc - updated).days if updated else None,
        "Status":            "🟢 Online" if connected else "🔴 Offline",
    })

df_gw = pd.DataFrame(gw_rows)
print(f"✅ {len(df_gw)} gateway(s)")
print(f"   Online: {df_gw['Conectado'].sum()} | Offline: {(~df_gw['Conectado']).sum()}")
df_gw


In [ ]:
# ── GET /v1/nextgensensor (comissionados) ─────
ns_resp = requests.get(f"{BASE_URL}/v1/nextgensensor",
                       headers=headers(), timeout=15)
ns_raw  = ns_resp.json() if ns_resp.status_code == 200 else []
ns_all  = ns_raw if isinstance(ns_raw, list) else ns_raw.get("value", [])
ns_list = [s for s in ns_all
           if (s.get("Commissioned") if s.get("Commissioned") is not None
               else s.get("commissioned"))]
print(f"✅ {len(ns_list)} comissionado(s) de {len(ns_all)} total")

sensor_rows = []
for s in ns_list:
    updated  = _parse_dt(s.get("StatusLastUpdated") or s.get("statusLastUpdated"))
    bat      = s.get("BatteryLevel") or s.get("batteryLevel")
    try:     bat = float(bat)
    except:  bat = None

    diag     = s.get("DiagnosticCode") or s.get("diagnosticCode") or 0
    try:     diag = int(diag)
    except:  diag = 0
    flags    = [lbl for bit, lbl in DIAG_BITS.items() if diag & bit]

    conn_raw = s.get("ConnectionState") or s.get("connectionState")
    try:     conn_code = int(conn_raw) if conn_raw is not None else None
    except:  conn_code = None
    conn_lbl = CONN_STATE_LABELS.get(conn_code, f"? ({conn_raw})")

    dias_off = (today_utc - updated).days if updated else None

    if conn_code == 0 or (dias_off and dias_off > 2):   alert = "danger"
    elif bat is not None and bat < 20:                    alert = "danger"
    elif conn_code in (2, 3):                             alert = "warn"
    elif diag & 512:                                      alert = "warn"
    elif bat is not None and bat < 40:                    alert = "warn"
    else:                                                 alert = "ok"

    sensor_rows.append({
        "IDNode":          s.get("IDNode") or s.get("idNode"),
        "HardwareID":      s.get("SensorIdentifier") or "—",
        "GatewayID":       s.get("IDSmartGateway") or s.get("idSmartGateway"),
        "ConnectionState": conn_lbl,
        "BatteryLevel":    bat,
        "LastUpdate":      updated.strftime("%Y-%m-%d %H:%M:%S") if updated else None,
        "DiasOffline":     dias_off,
        "DiagFlags":       ", ".join(flags) if flags else "—",
        "Alert":           alert,
    })

df_sens = pd.DataFrame(sensor_rows)
print(f"   Conectados:   {(df_sens['ConnectionState'] == 'Conectado').sum()}")
print(f"   Desconectados:{(df_sens['ConnectionState'] == 'Desconectado').sum()}")
print(f"   Sem medição:  {df_sens['ConnectionState'].isin(['Sem Medição','Conectado — Sem Medição']).sum()}")
print(f"   Bat < 20%:    {(df_sens['BatteryLevel'].fillna(100) < 20).sum()}")
df_sens.head(10)


In [ ]:
# ── Plot: ConnectionState (barras) ───────────
cs_counts = df_sens["ConnectionState"].value_counts().reset_index()
cs_counts.columns = ["Estado", "Qtd"]
color_map = {
    "Conectado":              C_OK,
    "Desconectado":           C_DANGER,
    "Sem Medição":            C_WARN,
    "Conectado — Sem Medição": C_PURPLE,
}
bar_colors = [color_map.get(s, C_TEAL) for s in cs_counts["Estado"]]

fig = go.Figure(go.Bar(
    x=cs_counts["Estado"], y=cs_counts["Qtd"],
    marker=dict(color=bar_colors, line=dict(color="rgba(255,255,255,0.08)", width=0.5)),
    text=cs_counts["Qtd"], textposition="outside",
    textfont=dict(size=11, color=C_TEXT),
    hovertemplate="<b>%{x}</b><br>%{y} sensor(es)<extra></extra>",
))
layout = PLOTLY_LAYOUT.copy()
layout.update(dict(
    title=dict(text="<b>Estado de Conexão dos Sensores</b> (OPC UA)",
               font=dict(size=16, color=C_TEXT), x=0.01),
    yaxis=dict(**PLOTLY_LAYOUT["yaxis"],
               title=dict(text="Sensores", font=dict(color=C_TEXT_MID))),
    height=320, showlegend=False,
))
fig.update_layout(**layout)
fig.show()


In [ ]:
# ── Plot: Scatter Bateria × Dias Offline ──────
df_sc = df_sens.dropna(subset=["BatteryLevel"]).copy()
df_sc["DiasOffline"] = df_sc["DiasOffline"].fillna(0)
ALERT_COLORS = {"ok": C_OK, "warn": C_WARN, "danger": C_DANGER}
colors = [ALERT_COLORS.get(a, C_TEAL) for a in df_sc["Alert"]]

fig = go.Figure()
fig.add_hrect(y0=0,  y1=20,  fillcolor=f"rgba(240,106,34,0.07)", line_width=0)
fig.add_hrect(y0=20, y1=40,  fillcolor=f"rgba(186,148,75,0.05)", line_width=0)
fig.add_hrect(y0=40, y1=100, fillcolor=f"rgba(78,157,45,0.03)",  line_width=0)
fig.add_vline(x=2, line=dict(color=f"rgba(240,106,34,0.4)", width=1, dash="dot"),
              annotation_text="2 dias", annotation_font=dict(color=C_DANGER, size=9))
fig.add_hline(y=20, line=dict(color=f"rgba(240,106,34,0.4)", width=1, dash="dot"),
              annotation_text="Bat 20%", annotation_font=dict(color=C_DANGER, size=9),
              annotation_position="bottom right")
fig.add_trace(go.Scatter(
    x=df_sc["DiasOffline"], y=df_sc["BatteryLevel"],
    mode="markers",
    marker=dict(size=11, color=colors,
                line=dict(color="rgba(255,255,255,0.15)", width=1)),
    customdata=df_sc[["HardwareID", "ConnectionState", "DiagFlags"]].values,
    hovertemplate="<b>%{customdata[0]}</b><br>Estado: %{customdata[1]}<br>"
                  "Bateria: %{y:.1f}%<br>Dias offline: %{x}<br>"
                  "Diagnóstico: %{customdata[2]}<extra></extra>",
))
layout = PLOTLY_LAYOUT.copy()
layout.update(dict(
    title=dict(text="<b>Saúde da Rede Mesh</b> · Bateria × Dias sem Atualização",
               font=dict(size=16, color=C_TEXT), x=0.01),
    xaxis=dict(**PLOTLY_LAYOUT["xaxis"],
               title=dict(text="Dias sem Atualização", font=dict(color=C_TEXT_MID))),
    yaxis=dict(**PLOTLY_LAYOUT["yaxis"], range=[0, 105],
               title=dict(text="Bateria [%]", font=dict(color=C_TEXT_MID))),
    height=400, hovermode="closest",
))
fig.update_layout(**layout)
fig.show()


---
## 7. Fleet — Dispositivos Cabeados & System Check

In [ ]:
# ── Constantes sync ──────────────────────────
SYNC_STATUS = {
    0: ("Não Sincronizado", "warn"),
    1: ("Sincronizado",     "ok"),
    2: ("Pendente",         "warn"),
    100: ("Falha",          "danger"),
}

# ── GET /v1/device ────────────────────────────
def get_wired_devices():
    resp = requests.get(f"{BASE_URL}/v1/device",
                        headers=headers(), timeout=15)
    if resp.status_code in (204, 404): return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

dev_raw  = get_wired_devices()
dev_rows = []
for d in dev_raw:
    updated   = _parse_dt(d.get("lastupdate") or d.get("LastUpdate"))
    sync_code = d.get("synchronizationstatus")
    try:   sync_code = int(sync_code)
    except: sync_code = None
    sync_lbl, sync_cls = SYNC_STATUS.get(sync_code, ("Desconhecido", "warn"))

    if sync_code == 100:       alert = "danger"
    elif sync_code == 2 and updated:
        alert = "warn" if (today_utc - updated).seconds // 60 > 30 else "ok"
    elif sync_code == 0:       alert = "warn"
    else:                      alert = "ok"

    dev_rows.append({
        "DeviceID":   d.get("id") or d.get("ID"),
        "Nome":       d.get("name") or d.get("Name") or "—",
        "Ativo":      bool(d.get("active") or d.get("Active")),
        "LastUpdate": updated.strftime("%Y-%m-%d %H:%M:%S") if updated else None,
        "DiasOffline":(today_utc - updated).days if updated else None,
        "SyncStatus": sync_lbl,
        "SyncCode":   sync_code,
        "Alert":      alert,
    })

df_dev = pd.DataFrame(dev_rows)
print(f"✅ {len(df_dev)} dispositivo(s) cabeado(s)")
df_dev


In [ ]:
# ── GET /v1/systemcheck ───────────────────────
def get_system_check():
    resp = requests.get(f"{BASE_URL}/v1/systemcheck",
                        headers=headers(), timeout=20)
    if resp.status_code in (204, 404): return []
    resp.raise_for_status()
    data = resp.json()
    return data if isinstance(data, list) else data.get("value", [])

sc_raw  = get_system_check()
sc_rows = []
for item in sc_raw:
    last_recv = _parse_dt(item.get("DateDataReceived") or item.get("dateDataReceived"))
    sc_rows.append({
        "Nome":             item.get("name") or item.get("Name") or "—",
        "Tipo":             item.get("type") or item.get("Type") or "—",
        "Falha":            item.get("error") or item.get("message") or "—",
        "DateDataReceived": last_recv.strftime("%Y-%m-%d %H:%M:%S") if last_recv else None,
        "DiasSemDados":     (today_utc - last_recv).days if last_recv else None,
    })

df_sc = pd.DataFrame(sc_rows)
print(f"✅ {len(df_sc)} item(ns) no system check")
df_sc.head(10)


In [ ]:
# ── Plot: SyncStatus (barras) ─────────────────
if not df_dev.empty:
    sync_counts = df_dev["SyncStatus"].value_counts().reset_index()
    sync_counts.columns = ["Status", "Qtd"]
    SYNC_COLORS = {
        "Sincronizado":     C_OK,
        "Pendente":         C_WARN,
        "Não Sincronizado": C_WARN,
        "Falha":            C_DANGER,
        "Desconhecido":     "#5C6670",
    }
    fig = go.Figure(go.Bar(
        x=sync_counts["Status"], y=sync_counts["Qtd"],
        marker=dict(color=[SYNC_COLORS.get(s, C_TEAL) for s in sync_counts["Status"]],
                    line=dict(color="rgba(255,255,255,0.08)", width=0.5)),
        text=sync_counts["Qtd"], textposition="outside",
        textfont=dict(size=11, color=C_TEXT),
        hovertemplate="<b>%{x}</b><br>%{y} dispositivo(s)<extra></extra>",
    ))
    layout = PLOTLY_LAYOUT.copy()
    layout.update(dict(
        title=dict(text="<b>Sincronização — Dispositivos Cabeados</b>",
                   font=dict(size=16, color=C_TEXT), x=0.01),
        yaxis=dict(**PLOTLY_LAYOUT["yaxis"],
                   title=dict(text="Qtd.", font=dict(color=C_TEXT_MID))),
        height=300, showlegend=False,
    ))
    fig.update_layout(**layout)
    fig.show()
else:
    print("ℹ Nenhum dispositivo cabeado encontrado.")


In [ ]:
# ── Resumo de alertas da frota ────────────────
criticals, warnings = [], []

if not df_gw.empty:
    n = int((~df_gw["Conectado"]).sum())
    if n: criticals.append(f"🔴 {n} gateway(s) OFFLINE")

if not df_sens.empty:
    n = int((df_sens["ConnectionState"] == "Desconectado").sum())
    if n: criticals.append(f"🚨 {n} sensor(es) Desconectado(s) (OPC UA 0)")
    n = int((df_sens["BatteryLevel"].fillna(100) < 20).sum())
    if n: criticals.append(f"🪫 {n} sensor(es) bateria < 20%")
    n = int(df_sens["ConnectionState"].isin(["Sem Medição","Conectado — Sem Medição"]).sum())
    if n: warnings.append(f"⚠ {n} sensor(es) Sem Medição (OPC UA 2/3)")
    n = int((df_sens["DiagFlags"].str.contains("Instabilidade", na=False)).sum())
    if n: warnings.append(f"📶 {n} sensor(es) com Instabilidade de Rede (bit 512)")

if not df_dev.empty:
    n = int((df_dev["SyncCode"] == 100).sum())
    if n: criticals.append(f"⛔ {n} dispositivo(s) com FALHA de sincronização")
    n = int((df_dev["SyncCode"] == 2).sum())
    if n: warnings.append(f"⏳ {n} dispositivo(s) com sincronização Pendente")

if not df_sc.empty:
    warnings.append(f"🔍 {len(df_sc)} item(ns) com falha no System Check")

print("=" * 50)
print("RESUMO DE ALERTAS DA FROTA")
print("=" * 50)
if not criticals and not warnings:
    print("✅ Frota dentro dos parâmetros normais.")
else:
    for msg in criticals: print(f"  CRÍTICO: {msg}")
    for msg in warnings:  print(f"  AVISO  : {msg}")
